<a href="https://colab.research.google.com/github/TalCordova/RMBA_SemB26_TC_SC/blob/main/Final_Project_SC_TC_fit_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Training Notebook

**Research Methods for Business Analytics** | Semester B, 2026 | Coller School of Management, Tel Aviv University

**Authors:**
- Saar Cohen, 213499312
- Tal Cordova, 203868187

---

## Overview

This notebook trains and evaluates classification models to predict whether a customer will purchase a product (`BUYER_FLAG = 1`), so the business can decide whom to target with a marketing contact.

It applies the feature engineering decisions and modelling strategy derived from the [EDA Notebook](Course_Final_Project_TC_SC.ipynb).

**Pipeline:**
1. **Feature engineering** — recency-weighted aggregation of yearly fare/points; drop uninformative flags
2. **Custom profit metric** — reflects the asymmetric payoff: TP = +$78.4, FP = −$15.2
3. **Baseline model comparison** — Logistic Regression, Decision Tree, Random Forest, XGBoost evaluated with 5-fold stratified CV

In [2]:
import pandas as pd
from sklearn.metrics import make_scorer

In [ ]:
# --- Mount Google Drive - run only in Google Colab ---
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
# -- Run this cell only if you are using Google Colab ---
tal_path_ffp_train = '/content/drive/MyDrive/PhD - TAU/Research Methods for Business Analytics/ffp_train.csv'
tal_path_reviews_train = '/content/drive/MyDrive/PhD - TAU/Research Methods for Business Analytics/reviews_training.csv'

saar_path_ffp_train ='/content/drive/MyDrive/Research Methods for Business Analytics/ffp_train.csv'
saar_pth_ffp_reviews = '/content/drive/MyDrive/Research Methods for Business Analytics/reviews_training.csv'
# --- Load training data for EDA ---
ffp_train = pd.read_csv(tal_path_ffp_train)
reviews_train = pd.read_csv(tal_path_reviews_train)

# --- Initial shape check ---
print(f'ffp_train shape: {ffp_train.shape}')
print(f'reviews_train shape: {reviews_train.shape}')

In [3]:
## load local
# --- Load training data for EDA ---
ffp_train = pd.read_csv('ffp_train.csv')
reviews_train = pd.read_csv('reviews_training.csv')

# --- Initial shape check ---
print(f'ffp_train shape: {ffp_train.shape}')
print(f'reviews_train shape: {reviews_train.shape}')


ffp_train shape: (20000, 23)
reviews_train shape: (1499, 2001)


In [4]:
ffp_rollout = pd.read_csv('ffp_rollout_X.csv')
reviews_rollout = pd.read_csv('reviews_rollout.csv')

print(f'ffp_rollout shape: {ffp_rollout.shape}')
print(f'reviews_rollout shape: {reviews_rollout.shape}')

ffp_rollout shape: (20000, 23)
reviews_rollout shape: (1501, 2001)


## 1. Feature Engineering

We apply two transformations informed by the EDA:

1. **Recency-weighted aggregation of `FARE` and `POINTS`** — the five yearly columns are highly collinear (r > 0.92). Rather than drop them or use a simple mean, we compute a weighted sum that gives more weight to recent years (Y1 = 1.0 down to Y5 = 0.2). This preserves magnitude while reducing the feature space from 10 columns to 2.

2. **Drop `ANIMAL_FLAG`** — the EDA showed it has virtually no discriminative power (near-zero lift and negligible correlation with the target).

In [5]:
# --- Feature Engineering: Recency-Weighted Fare & Points ---
RECENCY_WEIGHTS = [1.0, 0.8, 0.6, 0.4, 0.2]  # Y1 (most recent) → Y5 (oldest)

fare_cols   = [f'FARE_L_Y{i}'   for i in range(1, 6)]
points_cols = [f'POINTS_L_Y{i}' for i in range(1, 6)]

ffp_train['FARE_WEIGHTED']   = sum(
    w * ffp_train[c] for w, c in zip(RECENCY_WEIGHTS, fare_cols)
)
ffp_train['POINTS_WEIGHTED'] = sum(
    w * ffp_train[c] for w, c in zip(RECENCY_WEIGHTS, points_cols)
)

# --- Impute negative user grade with the median of all other user grades ---
median_grade = ffp_train.loc[ffp_train['CUSTOMER_GRADE'] >= 0, 'CUSTOMER_GRADE'].median()
ffp_train.loc[ffp_train['CUSTOMER_GRADE'] < 0, 'CUSTOMER_GRADE'] = median_grade

# --- Drop original yearly columns and ANIMAL_FLAG ---
cols_to_drop = fare_cols + points_cols + ['ANIMAL_FLAG']
ffp_train.drop(columns=cols_to_drop, inplace=True)

# --- Also drop EDA-only aggregates created earlier (FARE_MEAN, POINTS_MEAN) ---
ffp_train.drop(columns=['FARE_MEAN', 'POINTS_MEAN'], inplace=True, errors='ignore')

# --- Verify final feature set ---
print('Remaining columns:', ffp_train.columns.tolist())
print('Shape:', ffp_train.shape)

Remaining columns: ['ID', 'CUSTOMER_GRADE', 'STATUS_PANTINUM', 'STATUS_GOLD', 'STATUS_SILVER', 'NUM_DEAL', 'LAST_DEAL', 'ADVANCE_PURCHASE', 'ATT_FLAG', 'CREDIT_FLAG', 'RELATED_FLAG', 'BUYER_FLAG', 'FARE_WEIGHTED', 'POINTS_WEIGHTED']
Shape: (20000, 14)


## 2. Custom Profit Metric

The dataset is heavily imbalanced (~9.5:1 negative-to-positive ratio), and the business payoff is asymmetric:

| Outcome | Description | Value |
|---------|-------------|-------|
| True Positive (TP) | Contact a buyer → sale | **+$78.4** |
| False Positive (FP) | Contact a non-buyer → wasted outreach | **−$15.2** |
| True Negative / False Negative | No contact | $0 |

Standard accuracy or AUC do not capture this asymmetry. We therefore define a **per-customer average profit** score and optimise for it directly.

The **classification threshold** (0.1624) was derived from the TP/FP payoff ratio: a contact is profitable if `P(buyer) ≥ 15.2 / (15.2 + 78.4) ≈ 0.162`.

In [ ]:
THRESHOLD = 0.1624

feature_cols = [c for c in ffp_train.columns if c not in ['ID', 'BUYER_FLAG']]

X = ffp_train[feature_cols]
y = ffp_train['BUYER_FLAG']

# --- Custom profit scorer ---
def avg_profit(y_true, y_prob, threshold=THRESHOLD):
    y_pred = (y_prob >= threshold).astype(int)
    TP = ((y_pred == 1) & (y_true == 1)).sum()
    FP = ((y_pred == 1) & (y_true == 0)).sum()
    return (TP * 78.4 - FP * 15.2) / len(y_true)

profit_scorer = make_scorer(avg_profit, response_method='predict_proba')

## 3. Baseline Model Comparison

We evaluate four model families with 5-fold stratified cross-validation. Stratified folds preserve the 9.5% minority class proportion in every split.

Each model is wrapped in a `Pipeline`. Logistic Regression includes a `StandardScaler`; tree-based models do not need one.

Metrics reported per fold:
- **Avg Profit** — the primary metric (per-customer profit at threshold 0.1624)
- **AUC** — threshold-independent discrimination measure
- **Accuracy** — included for reference, but misleading given class imbalance

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.metrics import make_scorer, roc_auc_score, accuracy_score
import numpy as np

# --- Constants ---
THRESHOLD = 0.1624
RANDOM_STATE = 42

feature_cols = [c for c in ffp_train.columns if c not in ['ID', 'BUYER_FLAG']]
X = ffp_train[feature_cols]
y = ffp_train['BUYER_FLAG']

# --- Custom profit scorer ---
def avg_profit_scorer(y_true, y_prob, threshold=THRESHOLD):
    y_pred = (y_prob >= threshold).astype(int)
    TP = ((y_pred == 1) & (y_true == 1)).sum()
    FP = ((y_pred == 1) & (y_true == 0)).sum()
    return (TP * 78.4 - FP * 15.2) / len(y_true)

profit_scorer = make_scorer(avg_profit_scorer, response_method='predict_proba')
auc_scorer    = make_scorer(roc_auc_score, response_method='predict_proba')

scoring = {
    'avg_profit': profit_scorer,
    'auc':        auc_scorer,
    'accuracy':   'accuracy',
}

# --- CV strategy: stratified to preserve 9.5% minority class in each fold ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# --- Model definitions ---
# LR needs scaling; trees do not — each wrapped in its own Pipeline
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'Decision Tree': Pipeline([
        ('clf', DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    'Random Forest': Pipeline([
        ('clf', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))
        # n_jobs=-1 omitted — causes deadlocks in Colab
    ]),
    'XGBoost': Pipeline([
        ('clf', XGBClassifier(
            n_estimators=100,
            eval_metric='logloss',
            random_state=RANDOM_STATE,
            verbosity=0
        ))
    ]),
}

# --- Run CV for all models ---
results = {}

for name, pipeline in models.items():
    print(f'Running CV: {name}...')
    cv_results = cross_validate(
        pipeline, X, y,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )
    results[name] = {
        'avg_profit_mean': cv_results['test_avg_profit'].mean(),
        'avg_profit_std':  cv_results['test_avg_profit'].std(),
        'auc_mean':        cv_results['test_auc'].mean(),
        'auc_std':         cv_results['test_auc'].std(),
        'accuracy_mean':   cv_results['test_accuracy'].mean(),
        'accuracy_std':    cv_results['test_accuracy'].std(),
    }
    print(f"  Avg Profit: {results[name]['avg_profit_mean']:.4f} ± {results[name]['avg_profit_std']:.4f}")
    print(f"  AUC:        {results[name]['auc_mean']:.4f} ± {results[name]['auc_std']:.4f}")
    print(f"  Accuracy:   {results[name]['accuracy_mean']:.4f} ± {results[name]['accuracy_std']:.4f}")
    print()

# --- Summary table ---
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('avg_profit_mean', ascending=False)
print('=== Baseline Model Comparison (sorted by Avg Profit) ===')
print(results_df.round(4))

Running CV: Logistic Regression...
  Avg Profit: 0.4170 ± 0.1165
  AUC:        0.6528 ± 0.0190
  Accuracy:   0.9050 ± 0.0000

Running CV: Decision Tree...
  Avg Profit: 0.0703 ± 0.0970
  AUC:        0.5481 ± 0.0064
  Accuracy:   0.8313 ± 0.0021

Running CV: Random Forest...
  Avg Profit: 0.9614 ± 0.2024
  AUC:        0.6483 ± 0.0226
  Accuracy:   0.9037 ± 0.0015

Running CV: XGBoost...
  Avg Profit: 1.1269 ± 0.2420
  AUC:        0.6542 ± 0.0293
  Accuracy:   0.9018 ± 0.0023

=== Baseline Model Comparison (sorted by Avg Profit) ===
                     avg_profit_mean  avg_profit_std  auc_mean  auc_std  \
XGBoost                       1.1269          0.2420    0.6542   0.0293   
Random Forest                 0.9614          0.2024    0.6483   0.0226   
Logistic Regression           0.4170          0.1165    0.6528   0.0190   
Decision Tree                 0.0703          0.0970    0.5481   0.0064   

                     accuracy_mean  accuracy_std  
XGBoost                     0.9018  

## 4. Hyperparaeter Tuning

`XGBoost` is the best model on all metrics we tested (custom avergae user profit, AUC and accuracy) - so we will use it as our model.

In this section we will tune it.

### Why nested cross-validation?

The baseline comparison above reused the same 5 `StratifiedKFold` folds to score every model with **fixed** hyperparameters — there was no tuning involved, so no risk of optimism bias. If we now search over XGBoost's hyperparameters using those same 5 folds and report the best fold's score, the result would leak information: the folds used to *select* hyperparameters would be the same folds used to *report* performance, and the search would tend to find whatever configuration happens to fit those particular folds best (optimism bias / data leakage).

Nested cross-validation avoids this. An **outer** `StratifiedKFold` provides test folds that are never touched during the hyperparameter search; for each outer fold, an **inner** `StratifiedKFold` (run only on that fold's training data) drives a `RandomizedSearchCV`. The outer-fold scores then give an unbiased estimate of how the *tuning procedure* — not a single fixed model — performs on unseen data.

### Why optimize on `avg_profit`, not AUC?

AUC measures global ranking quality and is threshold-independent, but the business decision is **not** — we always classify at the fixed `THRESHOLD = 0.1624`. Hyperparameters such as `scale_pos_weight` shift the calibration of the predicted probabilities; a configuration that improves AUC can still push more probabilities above 0.1624 and hurt profit (more false positives), or vice versa. Since `avg_profit` is what the business actually cares about at that specific cutoff, the inner search refits on `avg_profit`; AUC and accuracy are still computed on every outer test fold and reported alongside it as diagnostics.

In [ ]:
# --- Nested CV: RandomizedSearchCV (inner) wrapped in cross_validate (outer) ---
from sklearn.model_selection import RandomizedSearchCV

xgb_pipeline = Pipeline([
    ('clf', XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE, verbosity=0))
])

xgb_param_dist = {
    'clf__n_estimators':     [100, 200, 300],
    'clf__max_depth':        [3, 4, 5, 6],
    'clf__learning_rate':    [0.01, 0.05, 0.1, 0.2],
    'clf__subsample':        [0.7, 0.8, 0.9, 1.0],
    'clf__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'clf__min_child_weight': [1, 3, 5, 7],
    'clf__gamma':            [0, 0.1, 0.5],
    'clf__scale_pos_weight': [1, 5, 9.5, 15],  # 9.5 ~= actual 18100/1900 imbalance ratio
}

# Separate from cell 11's `cv` (identical spec) so that cell stays independently re-runnable
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_dist,
    n_iter=20,
    scoring=scoring,
    refit='avg_profit',
    cv=inner_cv,
    random_state=RANDOM_STATE,
    n_jobs=1,  # n_jobs=-1 causes deadlocks in Colab (see Random Forest note above)
    verbose=0,
)

# cross_validate clones + fits xgb_search on each outer-train fold (running the full
# inner search inside it), then scores the refit best_estimator_ on the outer-test fold.
nested_cv_results = cross_validate(
    xgb_search, X, y,
    cv=outer_cv,
    scoring=scoring,
    return_estimator=True,
    n_jobs=1,
)

print('Best hyperparameters per outer fold:')
for fold_idx, fitted_search in enumerate(nested_cv_results['estimator']):
    print(f'  Fold {fold_idx}: {fitted_search.best_params_}')

results['XGBoost (Tuned, Nested CV)'] = {
    'avg_profit_mean': nested_cv_results['test_avg_profit'].mean(),
    'avg_profit_std':  nested_cv_results['test_avg_profit'].std(),
    'auc_mean':        nested_cv_results['test_auc'].mean(),
    'auc_std':         nested_cv_results['test_auc'].std(),
    'accuracy_mean':   nested_cv_results['test_accuracy'].mean(),
    'accuracy_std':    nested_cv_results['test_accuracy'].std(),
}

results_df = pd.DataFrame(results).T.sort_values('avg_profit_mean', ascending=False)
print('\n=== Updated Model Comparison (sorted by Avg Profit) ===')
print(results_df.round(4))

### Results interpretation (from a local verification run)

Running this section locally (sklearn 1.6.1, xgboost 3.3.0) produced:

| | avg_profit | AUC | accuracy |
|---|---|---|---|
| XGBoost (Tuned, Nested CV) | 1.59 ± 0.25 | 0.671 ± 0.023 | 0.905 ± 0.001 |
| XGBoost (baseline, untuned) | 1.00 ± 0.16 | 0.650 ± 0.029 | 0.900 ± 0.003 |

Tuning lifted average profit by roughly **+59%** over the untuned baseline (1.00 → 1.59), with AUC also improving (0.650 → 0.671) — both metrics moved in the same direction, so the profit gain isn't just an artifact of probability recalibration crossing the fixed 0.1624 threshold. (Exact numbers will shift slightly when re-run on Colab, since XGBoost's tree construction isn't bit-identical across library versions even with a fixed `random_state` — the *direction and rough magnitude* of the improvement is what matters.)

All 5 outer folds' inner searches converged on the **same** hyperparameters: shallower trees (`max_depth=3` vs. the untuned default of 6), a lower `learning_rate` (0.05 vs. default 0.3), row/column subsampling (`subsample=0.8`, `colsample_bytree=0.7`), and — notably — `scale_pos_weight=1`, i.e. **no** extra reweighting for the 9.5:1 class imbalance. That the search consistently avoided reweighting suggests XGBoost's own gradient boosting already handles this imbalance reasonably well at the chosen threshold, and that explicit reweighting mostly just shifts the operating point without a net profit benefit here. All 5 folds agreeing (rather than scattering across the search space) is itself a good sign — the selection isn't noise-driven.

In [ ]:
# --- Final hyperparameter search on full data (the model to carry forward) ---
final_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_dist,
    n_iter=20,
    scoring=scoring,
    refit='avg_profit',
    cv=outer_cv,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=0,
)
final_search.fit(X, y)

print('Final selected hyperparameters (XGBoost):')
for k, v in final_search.best_params_.items():
    print(f'  {k}: {v}')
print(f"Inner CV avg_profit at these params: {final_search.best_score_:.4f}")

xgb_final_model = final_search.best_estimator_

In [ ]:
# --- Custom asymmetric-loss XGBoost: bake the $78.4/$15.2 cost ratio into the gradient ---
from scipy.special import expit

FN_OVER_FP_RATIO = 78.4 / 15.2  # ~5.158 -- how much harder a missed buyer is penalized than a wasted contact

def asymmetric_profit_objective(y_true, y_pred):
    # y_pred is the raw margin (pre-sigmoid); XGBoost custom objectives operate in margin space.
    p = expit(y_pred)
    grad = (1 - y_true) * p - FN_OVER_FP_RATIO * y_true * (1 - p)
    hess = ((1 - y_true) + FN_OVER_FP_RATIO * y_true) * p * (1 - p)
    return grad, hess

# Freeze the nested-CV-selected hyperparameters from xgb_final_model, swap in only the objective
frozen_params = xgb_final_model.named_steps['clf'].get_params()
frozen_params['objective'] = asymmetric_profit_objective

custom_loss_pipeline = Pipeline([
    ('clf', XGBClassifier(**frozen_params))
])

# Standard (non-nested) 5-fold CV — no hyperparameter search here, just re-scoring a fixed config
custom_loss_cv_results = cross_validate(
    custom_loss_pipeline, X, y,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

results['XGBoost (Custom Asymmetric Loss)'] = {
    'avg_profit_mean': custom_loss_cv_results['test_avg_profit'].mean(),
    'avg_profit_std':  custom_loss_cv_results['test_avg_profit'].std(),
    'auc_mean':        custom_loss_cv_results['test_auc'].mean(),
    'auc_std':         custom_loss_cv_results['test_auc'].std(),
    'accuracy_mean':   custom_loss_cv_results['test_accuracy'].mean(),
    'accuracy_std':    custom_loss_cv_results['test_accuracy'].std(),
}

results_df = pd.DataFrame(results).T.sort_values('avg_profit_mean', ascending=False)
print('=== Final Model Comparison (sorted by Avg Profit) ===')
print(results_df.round(4))

### Discussion: does embedding the cost ratio in the loss beat threshold-level tuning?

Running this locally (sklearn 1.6.1, xgboost 3.3.0) gave:

| | avg_profit | AUC | accuracy |
|---|---|---|---|
| XGBoost (Custom Asymmetric Loss) | **−6.25 ± 0.02** | **0.673 ± 0.023** | 0.848 ± 0.008 |
| XGBoost (Tuned, Nested CV) | 1.59 ± 0.25 | 0.671 ± 0.023 | 0.905 ± 0.001 |

The custom-loss model achieves the **best AUC of any model tested** — marginally above the nested-CV winner — confirming that weighting false negatives `FN_OVER_FP_RATIO` (≈5.158×) harder than false positives does sharpen the model's ranking of who is likely to buy. But average profit collapses to **−6.25**, far worse than every other model, including the untuned baseline. The reason is calibration, not ranking: a fitted check (in-sample) shows the model assigns `P(buyer) ≥ 0.1624` to 99.8% of all customers (it achieves near-perfect recall by simply predicting "buyer" almost everywhere), because the gradient now treats *any* false negative as ~5× more costly than a false positive, regardless of how rare buyers actually are. AUC is rank-based and invariant to this global shift; `avg_profit`, evaluated at the **fixed** threshold of 0.1624, is not — it implodes once nearly every contact is a false positive at −$15.2 apiece.

This is exactly the failure mode flagged earlier in this notebook: AUC and profit can move in opposite directions when a hyperparameter (or, here, the loss itself) recalibrates probabilities without the decision threshold moving with it. It also explains, in hindsight, why the nested search in Section 4 always selected `scale_pos_weight = 1` — both mechanisms inject the same kind of asymmetry, and the profit-aware search correctly rejected it because the threshold was never re-optimized to match. A custom loss could in principle still help, but only paired with a re-derived (or jointly tuned) decision threshold — out of scope here.

**Implication for deployment:** `xgb_final_model` (from the profit-optimized nested CV) remains the model to carry forward. It was selected by directly validating against `avg_profit` at the threshold we actually deploy at, rather than by a proxy (AUC) or an unconstrained cost-weighting that ignores the operating point.

In [ ]:
# --- Threshold optimization: can a recalibrated cutoff rescue the custom-loss model's profit? ---
from sklearn.model_selection import cross_val_predict

oof_proba = cross_val_predict(custom_loss_pipeline, X, y, cv=cv, method='predict_proba', n_jobs=1)[:, 1]

thresholds = np.round(np.linspace(0.10, 0.95, 86), 2)  # 0.10 to 0.95 in steps of 0.01
profits = np.array([avg_profit_scorer(y, oof_proba, threshold=t) for t in thresholds])

best_idx = int(np.argmax(profits))
best_threshold = float(thresholds[best_idx])
best_profit = float(profits[best_idx])

print(f'Best threshold: {best_threshold:.2f}')
print(f'Max avg_profit (pooled out-of-fold) at this threshold: {best_profit:.4f}')

# Recompute fold-level profit/accuracy at the optimized threshold for a results-table-consistent
# mean/std (AUC is threshold-independent, so it is carried over unchanged)
fold_profits, fold_accuracies = [], []
for _, test_idx in cv.split(X, y):
    y_te = y.iloc[test_idx].values
    p_te = oof_proba[test_idx]
    fold_profits.append(avg_profit_scorer(y_te, p_te, threshold=best_threshold))
    fold_accuracies.append(accuracy_score(y_te, (p_te >= best_threshold).astype(int)))

results['XGBoost (Custom Asymmetric Loss)'] = {
    'avg_profit_mean': np.mean(fold_profits),
    'avg_profit_std':  np.std(fold_profits),
    'auc_mean':        results['XGBoost (Custom Asymmetric Loss)']['auc_mean'],
    'auc_std':         results['XGBoost (Custom Asymmetric Loss)']['auc_std'],
    'accuracy_mean':   np.mean(fold_accuracies),
    'accuracy_std':    np.std(fold_accuracies),
}

results_df = pd.DataFrame(results).T.sort_values('avg_profit_mean', ascending=False)
print('\n=== Final Comparison (Custom Asymmetric Loss now using its optimized threshold) ===')
print(results_df.round(4))

### Final summary: two paradigms for handling the cost asymmetry

Running this locally (sklearn 1.6.1, xgboost 3.3.0):

| Paradigm | avg_profit | AUC | accuracy |
|---|---|---|---|
| 1. Standard loss + fixed threshold (0.1624) — `xgb_final_model` | 1.5934 ± 0.2453 | 0.6713 ± 0.0228 | 0.9046 ± 0.0007 |
| 2. Custom asymmetric loss + empirical threshold (0.50) | 1.5938 ± 0.2631 | 0.6728 ± 0.0229 | 0.8477 ± 0.0080 |

Once the decision threshold is recalibrated for the custom-loss model, its profit recovers completely — from −6.25 up to **1.5938**, statistically indistinguishable from paradigm 1's 1.5934 (the gap, 0.0004, is two orders of magnitude smaller than either model's standard deviation). This confirms the earlier diagnosis: the custom loss was never a *worse* model, it was a *miscalibrated* one, and a recalibrated cutoff closes essentially the entire gap. Interestingly, the data-driven optimum (0.50) lands almost exactly on XGBoost's conventional default classification threshold — a sanity-check-style result, not an arbitrary number.

The two paradigms are therefore a statistical tie on profit, but they are not equivalent on every axis:
- **Accuracy** is meaningfully higher under paradigm 1 (0.905 vs. 0.848) — paradigm 2 operates at a less conservative cutoff (13.5% of customers flagged vs. a 9.5% base rate), trading some precision for the recall the asymmetric gradient was built to reward.
- **Evaluation rigor** favors paradigm 1: its 0.1624 threshold was fixed *a priori* from the business payoff ratio and never touched by any data-driven search, and the nested CV that selected its hyperparameters never saw the data it was scored on. Paradigm 2's 1.5938 profit estimate required an *additional* threshold search performed on the same out-of-fold predictions later used to report that score — a much smaller-scale version of the optimism risk nested CV exists to eliminate, since it's a single scalar rather than an 8-dimensional hyperparameter space.
- **Operational simplicity** favors paradigm 1: it uses a standard, well-supported loss function and a cutoff that is directly traceable to the $78.4/$15.2 payoffs, rather than a custom gradient/Hessian function plus an empirically-fit cutoff that would need to be re-validated if the class balance or cost structure changes.

**Conclusion:** paradigm 2 is a successful proof-of-concept — it shows the AUC/profit divergence was purely a calibration artifact, recoverable by re-optimizing the threshold — but it does not deliver a profit advantage large enough to justify its added complexity and slightly weaker evaluation guarantees. We recommend keeping **`xgb_final_model`** (paradigm 1: standard loss, profit-tuned hyperparameters via nested CV, fixed business threshold) as the final deployable asset for the campaign.

## 5 Add Text Features

In this part, we will add the cell features from the [text mining notebook](C:\Users\talcordova\Projects\RMBA_SemB26_TC_SC\Course_Project_Text_Mining_TC_SC.ipynb) - we wil ladd two features:

1. `has_review` - a flag to indicate if a user has a review.
2. `positive_proba` - the probabiloty from the LR model that the user will have a positive review.

In [ ]:
# --- Add Text Features: has_review & positive_proba ---

review_preds = pd.read_csv('reviews_train_preds.csv')  # columns: ID, rating

# Map ID -> predicted positive probability
proba_by_id = review_preds.set_index('ID')['rating']

# has_review: 1 if the customer has a review prediction, else 0
ffp_train['has_review'] = ffp_train['ID'].isin(proba_by_id.index).astype(int)

# positive_proba: predicted probability for reviewers; -1 sentinel for the rest.

ffp_train['positive_proba'] = ffp_train['ID'].map(proba_by_id).fillna(-1)

# --- Sanity checks ---
n_reviewers = ffp_train['has_review'].sum()
print(f'Customers with a review: {n_reviewers} / {len(ffp_train)}')
print(f'positive_proba range (reviewers): '
      f"[{ffp_train.loc[ffp_train['has_review'] == 1, 'positive_proba'].min():.4f}, "
      f"{ffp_train.loc[ffp_train['has_review'] == 1, 'positive_proba'].max():.4f}]")
print(f"Non-reviewer positive_proba unique values: "
      f"{ffp_train.loc[ffp_train['has_review'] == 0, 'positive_proba'].unique()}")
ffp_train[['ID', 'has_review', 'positive_proba']].head()

In [ ]:
# --- Add Text Features to rollout ---

review_preds_rollout = pd.read_csv('reviews_rollout_preds.csv')  # columns: ID, rating

# Map ID -> predicted positive probability
proba_by_id = review_preds_rollout.set_index('ID')['rating']

# has_review: 1 if the customer has a review prediction, else 0
ffp_rollout['has_review'] = ffp_rollout['ID'].isin(proba_by_id.index).astype(int)

# positive_proba: predicted probability for reviewers; -1 sentinel for the rest.

ffp_rollout['positive_proba'] = ffp_rollout['ID'].map(proba_by_id).fillna(-1)

# --- Sanity checks ---
n_reviewers = ffp_rollout['has_review'].sum()
print(f'Customers with a review: {n_reviewers} / {len(ffp_rollout)}')
print(f'positive_proba range (reviewers): '
      f"[{ffp_rollout.loc[ffp_rollout['has_review'] == 1, 'positive_proba'].min():.4f}, "
      f"{ffp_rollout.loc[ffp_rollout['has_review'] == 1, 'positive_proba'].max():.4f}]")
print(f"Non-reviewer positive_proba unique values: "
      f"{ffp_rollout.loc[ffp_rollout['has_review'] == 0, 'positive_proba'].unique()}")
ffp_rollout[['ID', 'has_review', 'positive_proba']].head()